In [ ]:
### AGRIM KAPOOR 23 CS 032

In [1]:
!pip install kaggle

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import os

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform)

test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

num_classes = 10

100%|██████████| 170M/170M [00:04<00:00, 41.2MB/s]


In [4]:
class CNN(nn.Module):
    def __init__(self, activation='relu'):
        super(CNN, self).__init__()

        if activation == 'relu':
            self.activation = nn.ReLU()
        elif activation == 'tanh':
            self.activation = nn.Tanh()
        elif activation == 'leakyrelu':
            self.activation = nn.LeakyReLU(0.01)

        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool = nn.MaxPool2d(2, 2)

        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(self.activation(self.bn1(self.conv1(x))))
        x = self.pool(self.activation(self.bn2(self.conv2(x))))
        x = x.view(-1, 64 * 8 * 8)
        x = self.activation(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [5]:
def initialize_weights(model, method='xavier'):
    for m in model.modules():
        if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
            if method == 'xavier':
                nn.init.xavier_uniform_(m.weight)
            elif method == 'kaiming':
                nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
            elif method == 'random':
                nn.init.normal_(m.weight, mean=0, std=0.01)

In [6]:
def train_model(model, optimizer, epochs=5):
    criterion = nn.CrossEntropyLoss()
    model.to(device)

    best_acc = 0

    for epoch in range(epochs):
        model.train()
        running_loss = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        acc = evaluate(model)
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss:.4f}, Test Accuracy: {acc:.2f}%")

        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), "best_cnn_model.pth")

    return best_acc

In [7]:
def evaluate(model):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return 100 * correct / total

In [ ]:
activations = ['relu', 'tanh', 'leakyrelu']
initializations = ['xavier', 'kaiming', 'random']
optimizers_list = ['sgd', 'adam', 'rmsprop']

results = {}

for act in activations:
    for init in initializations:
        for opt in optimizers_list:

            print(f"\nTraining: Activation={act}, Init={init}, Optimizer={opt}")

            model = CNN(activation=act)
            initialize_weights(model, method=init)

            if opt == 'sgd':
                optimizer = optim.SGD(model.parameters(), lr=0.01)
            elif opt == 'adam':
                optimizer = optim.Adam(model.parameters(), lr=0.001)
            elif opt == 'rmsprop':
                optimizer = optim.RMSprop(model.parameters(), lr=0.001)

            acc = train_model(model, optimizer, epochs=5)

            results[(act, init, opt)] = acc


Training: Activation=relu, Init=xavier, Optimizer=sgd
Epoch [1/5], Loss: 1382.2419, Test Accuracy: 49.23%
Epoch [2/5], Loss: 1139.9506, Test Accuracy: 54.15%
Epoch [3/5], Loss: 1038.2280, Test Accuracy: 56.16%
Epoch [4/5], Loss: 969.1611, Test Accuracy: 56.05%
Epoch [5/5], Loss: 915.8721, Test Accuracy: 56.93%

Training: Activation=relu, Init=xavier, Optimizer=adam
Epoch [1/5], Loss: 1539.2197, Test Accuracy: 40.87%
Epoch [2/5], Loss: 1363.0485, Test Accuracy: 49.42%
Epoch [3/5], Loss: 1299.1003, Test Accuracy: 54.99%
Epoch [4/5], Loss: 1250.4735, Test Accuracy: 56.41%
Epoch [5/5], Loss: 1213.7109, Test Accuracy: 58.95%

Training: Activation=relu, Init=xavier, Optimizer=rmsprop
Epoch [1/5], Loss: 1656.1102, Test Accuracy: 30.09%
Epoch [2/5], Loss: 1349.2059, Test Accuracy: 41.71%
Epoch [3/5], Loss: 1278.5946, Test Accuracy: 54.62%
Epoch [4/5], Loss: 1244.3191, Test Accuracy: 55.71%
Epoch [5/5], Loss: 1216.1497, Test Accuracy: 52.95%

Training: Activation=relu, Init=kaiming, Optimizer=

In [ ]:
best_config = max(results, key=results.get)
print("\nBest Configuration:", best_config)
print("Best Accuracy:", results[best_config])

In [ ]:
resnet = models.resnet18(pretrained=True)

# Modify final layer
resnet.fc = nn.Linear(resnet.fc.in_features, num_classes)
resnet = resnet.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet.parameters(), lr=0.001)

train_model(resnet, optimizer, epochs=5)